# 06 Per-Cluster: 逐簇深度剖析

面向 **非计算机专业 PI/学生**（ADR-0009）。消费 `06_annotated.ipynb` 的输出，
对 `cell_type_final_v1` 的每个簇单独产出以下内容：

- **UMAP 高亮**（该簇在全局 UMAP 上的位置）
- **Top 标记基因 dotplot**（该簇 vs 其余簇的差异基因）
- **基因集评分小提琴图**（该簇细胞在各细胞类型评分上的分布）
- **跨疾病组丰度柱状图**（该簇在不同 disease/disease_group 中的比例）
- **LLM 叙述段落**（由 LLM 综合以上证据写一段该簇的生物学描述，key 守卫——无 key 跳过）

产物：`results/figures/06b_per_cluster/cluster_{label}.md` + `index.md`

**为什么逐簇做而不是一张总图？** 每个簇的生物学意义需要用文字讲清楚——
这张图解释它在组织中的位置，这段文字解释它的功能、marker 证据、
跨疾病的丰度差异。逐簇 markdown 可以直接作为论文 Results section
的初稿素材，而不是一张看不懂的热图。

**实现纪律（ADR-0003/0009）**：纯 for 循环，无 plugin/registry/class。
判据是非 CS 学生打开 notebook 能否逐行看懂。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **本 notebook 角色**：只读逐簇深度报告——消费 `06_annotated` 的输出，
  对 `cell_type_final_v1` 的每个簇产出独立 markdown 报告。
- **下游**：无（本 notebook 不产出 h5ad，仅产出 markdown 报告供 PI 审阅）

### 为什么要迭代回跑？
逐簇深度报告是 PI 审阅注释质量的主要入口。当 PI 在上游 06 调整了注释标签
（修改 `marker_assignments` / `pi_decisions`，或换了 `LEIDEN_COL`、`MARKER_CSV`）后，
需要重跑本 notebook 以生成反映最新注释的逐簇报告。

### 如何回跑（两步操作）
1. **改 `UPSTREAM_PATH`**——指向新版 06_annotated 输出
   （例如 `06_annotated_v2.h5ad`）
2. **（可选）改 `OUTPUT_DIR`**——指向新版逐簇报告目录
   （例如 `results/figures/06b_per_cluster_v2/`）避免覆盖旧版报告
   → 重跑本 notebook（Cell → Run All）

### 重要说明
- **本 notebook 是只读分析，不产出 h5ad，不写入 `adata.uns`。**
  因此没有 `stage` / `version` / `upstream` / `status` 追踪字段——这些字段由上游
  `06_annotated` 负责维护。
- 如需追溯"这份逐簇报告是用哪个版本的注释跑的"，检查本 notebook 的
  `UPSTREAM_PATH` 参数，然后查看对应 `.h5ad` 的 `adata.uns["stage"]` 和
  `adata.uns["version"]`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH    -- 06_annotated 输出 h5ad（必须有 cell_type_final_v1 列）
# OUTPUT_DIR       -- 逐簇 markdown 输出目录
# LABEL_COL        -- 用哪个 obs 列做逐簇分析（默认 cell_type_final_v1）
# GENESET_CSV      -- 基因集评分用的标记物 CSV（同 06_annotated 的 MARKER_CSV）
# DISEASE_COL      -- 跨疾病丰度分析用哪个 obs 列（如 disease / disease_group）
# N_TOP_GENES      -- 每簇展示 Top 几个标记基因
# VERDICT_MODEL    -- LLM 叙述段落用的模型（同 06_annotated 的 VERDICT_MODEL）
# BASE_URLS        -- provider base URLs（同 06_annotated）

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
OUTPUT_DIR    = "results/figures/06b_per_cluster"
LABEL_COL     = "cell_type_final_v1"
GENESET_CSV   = "references/markers/gastric_epithelial.csv"
DISEASE_COL   = "disease"  # 或 "disease_group"，取决于上游 manifest 的 obs 列
N_TOP_GENES   = 15

# LLM 叙述段落模型（key 守卫——无 key 跳过）
VERDICT_MODEL = "anthropic/claude-sonnet-4-6"

# 各家 provider 的 base URL（mLLMCelltype 按模型名前缀自动路由）
BASE_URLS = {
    "openai":     "",
    "anthropic":  "",
    "deepseek":   "https://api.deepseek.com",
    "qwen":       "",
    "gemini":     "",
}

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 导入（scanpy 原生 + 框架函数）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime, warnings

from scrna_integration import load_markers

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 加载上游 adata
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 列出 obs 中有用的列
_cell_type_cols = [c for c in adata.obs.columns if "cell_type" in c or "leiden" in c]
_score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
print(f"细胞类型相关列: {_cell_type_cols}")
print(f"基因集评分列: {_score_cols}")

## 验证标签列 + 准备输出目录

**为什么先验证？** 如果 LABEL_COL 不存在，直接报错比跑一半才发现更友好。
同时确保标签列是 categorical 类型——scanpy 的 `cat.categories` 依赖这个。

In [ ]:
# 验证标签列存在且非空。
if LABEL_COL not in adata.obs.columns:
    raise KeyError(
        f"LABEL_COL='{LABEL_COL}' 不在 adata.obs.columns 中。"
        f"可用列: {sorted(adata.obs.columns.tolist())}"
    )

# 确保是 categorical（否则 .cat.categories 不可用）
if not hasattr(adata.obs[LABEL_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LABEL_COL]):
    adata.obs[LABEL_COL] = adata.obs[LABEL_COL].astype("category")

_cluster_ids = sorted(adata.obs[LABEL_COL].cat.categories)
# 排除 NaN 类别（PI 可能部分簇未标注）
_cluster_ids = [c for c in _cluster_ids if pd.notna(c) and str(c) != "nan"]
_cluster_sizes = {c: (adata.obs[LABEL_COL] == c).sum() for c in _cluster_ids}

print(f"标签列: {LABEL_COL}")
print(f"有效簇数: {len(_cluster_ids)}")
for c in _cluster_ids:
    print(f"  {c}: {_cluster_sizes[c]:,} 细胞")

# 加载标记物库（供基因集评分）
_markers = load_markers(GENESET_CSV)
print(f"\n标记物库: {GENESET_CSV} ({len(_markers)} 种细胞类型)")

# 确认上游 embedding 存在（UMAP 绘图需要）
if "X_umap" not in adata.obsm:
    print("\n⚠ 警告: adata.obsm 中无 X_umap，UMAP 高亮图将跳过。"
          "请确保上游 04 已计算 UMAP 坐标。")

# 确认 DISEASE_COL 存在
_disease_available = DISEASE_COL in adata.obs.columns
if _disease_available:
    _diseases = sorted(adata.obs[DISEASE_COL].dropna().unique())
    print(f"\n疾病列 '{DISEASE_COL}' 可用: {len(_diseases)} 种 ({_diseases})")
else:
    print(f"\n⚠ 疾病列 '{DISEASE_COL}' 不存在——跨疾病丰度图将跳过。"
          f"可用 obs 列: {sorted(adata.obs.columns.tolist())}")

## 逐簇循环体

对每个细胞类型标签，执行以下步骤生成一份 markdown：

1. **UMAP 高亮**：`sc.pl.umap` 加 `groups=` 参数，高亮该簇细胞在全局 UMAP 中的位置
2. **Top 标记基因**：`sc.tl.rank_genes_groups` 找该簇 vs 其余簇的差异基因，画 dotplot
3. **基因集评分小提琴**：将该簇细胞的 `score_{celltype}` 列画小提琴图
4. **跨疾病丰度**：将该簇在不同疾病组中的细胞比例画柱状图
5. **LLM 叙述**（可选，key 守卫）：调用 LLM 综合以上证据写一段生物学描述

**为什么在循环内调 `sc.tl.rank_genes_groups`？**
整个 adata 已在上游 06 做了全局 rank_genes_groups。
但这里我们要的是 **该簇 vs 其余所有簇** 的对比（用 `reference="rest"`），
每次子集 mask 不同，所以必须在循环内独立调用。
这是 scanpy 原生操作——没有额外抽象。

In [ ]:
# 逐簇剖析：for 循环遍历每个细胞类型标签。
# 为什么用 for 循环而非 sweep/plugin？见 ADR-0003/0009——
# 非 CS 学生一眼看懂 for 循环，但畏惧回调/注册中心。
import json

# 预处理：如果有 score 列，收集列名
_score_cols_all = [c for c in adata.obs.columns if c.startswith("score_")]
# 确保至少有一个基础 embedding 可用于 UMAP
_has_umap = "X_umap" in adata.obsm

# 如果无 UMAP，尝试在当前 obsm 中找一个 X_ 开头的做 fallback
_umap_key = "X_umap"
if not _has_umap:
    _umap_keys = [k for k in adata.obsm.keys() if k.startswith("X_")]
    if _umap_keys:
        _umap_key = _umap_keys[0]
        _has_umap = True
        print(f"UMAP 不可用，改用 {_umap_key} 坐标画散点图（非标准 UMAP 投影）\n")

# 收集每簇的一行摘要，用于 index.md
_cluster_summaries = []

for _cid in _cluster_ids:
    _cid_str = str(_cid).replace("/", "_").replace(" ", "_")
    _mask = adata.obs[LABEL_COL] == _cid
    _n = _mask.sum()
    _out_md = os.path.join(OUTPUT_DIR, f"cluster_{_cid_str}.md")
    print(f"\n{'='*60}")
    print(f"处理簇 '{_cid}': {_n:,} 细胞")
    print(f"{'='*60}")

    # ---- 打开该簇的 markdown 输出文件 ----
    _md_lines = []
    _md_lines.append(f"# 簇: {_cid}\n")
    _md_lines.append(f"**标签列**: `{LABEL_COL}`\n")
    _md_lines.append(f"**细胞数**: {_n:,} ({_n/adata.n_obs*100:.1f}%)\n")
    _md_lines.append(f"**上游**: `{UPSTREAM_PATH}`\n")

    # ---- 1. UMAP 高亮 ----
    _md_lines.append(f"\n## 1. UMAP 高亮\n")
    if _has_umap:
        # 创建 highlight 列：该簇为实际标签，其余为 "Other"
        # 为什么用 copy 新列？不改动原始 adata.obs，隔离副作用
        _highlight_col = f"_highlight_{_cid_str}"
        adata.obs[_highlight_col] = "Other"
        adata.obs.loc[_mask, _highlight_col] = str(_cid)
        # 该簇的颜色用红色突出，其余灰色
        _palette = {str(_cid): "#e74c3c", "Other": "#bdc3c7"}

        _fig, _ax = plt.subplots(figsize=(10, 8))
        sc.pl.umap(
            adata, color=_highlight_col, ax=_ax,
            palette=_palette, title=f"UMAP: {_cid} ({_n:,} cells)",
            legend_loc="right margin", show=False,
        )
        _fname = os.path.join(OUTPUT_DIR, f"umap_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)
        _md_lines.append(f"![UMAP {_cid}](umap_{_cid_str}.png)\n")
        # 清理临时列
        del adata.obs[_highlight_col]
        print(f"  UMAP 高亮已保存: {_fname}")
    else:
        _md_lines.append("(UMAP 坐标不可用，跳过)\n")
        print("  UMAP 跳过: 无可用的嵌入坐标")

    # ---- 2. Top 标记基因 dotplot ----
    _md_lines.append(f"\n## 2. Top 标记基因\n")
    # 在 adata 上用 mask 做 rank_genes_groups
    # 注意：rank_genes_groups 会写入 adata.uns，每簇覆盖前一次的 key
    sc.tl.rank_genes_groups(
        adata, groupby=LABEL_COL, groups=[_cid],
        reference="rest", method="wilcoxon", n_genes=N_TOP_GENES,
        key_added="rank_genes_per_cluster",
    )
    _df = sc.get.rank_genes_groups_df(adata, group=_cid, key="rank_genes_per_cluster")
    _top_genes = _df["names"].head(N_TOP_GENES).tolist()
    _top_scores = _df["scores"].head(N_TOP_GENES).tolist()
    _top_logfc = _df["logfoldchanges"].head(N_TOP_GENES).tolist()
    _top_pvals = _df["pvals_adj"].head(N_TOP_GENES).tolist()

    _md_lines.append(f"_Top {len(_top_genes)} 差异基因（Wilcoxon, {_cid} vs rest）_\n\n")
    _md_lines.append("| 排名 | 基因 | logFC | -log10(p_adj) |\n")
    _md_lines.append("|------|------|-------|---------------|\n")
    for i, (g, lfc, p) in enumerate(zip(_top_genes, _top_logfc, _top_pvals)):
        _nlogp = -np.log10(max(p, 1e-300))
        _md_lines.append(f"| {i+1} | {g} | {lfc:.2f} | {_nlogp:.1f} |\n")

    # 画 dotplot（该簇 vs 其余，用 top 基因）
    if _top_genes and _has_umap:
        _fig, _ax = plt.subplots(figsize=(max(6, len(_top_genes)*0.4), 3))
        sc.pl.dotplot(
            adata, var_names=_top_genes, groupby=LABEL_COL,
            standard_scale="var", title=f"Top markers: {_cid}",
            show=False, ax=_ax,
        )
        _fname = os.path.join(OUTPUT_DIR, f"dotplot_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)
        _md_lines.append(f"\n![Dotplot {_cid}](dotplot_{_cid_str}.png)\n")
        print(f"  Top 标记基因: {_top_genes}")
    else:
        print(f"  Top 标记基因: {_top_genes} (无 UMAP，跳过 dotplot)")

    # ---- 3. 基因集评分小提琴图 ----
    _md_lines.append(f"\n## 3. 基因集评分\n")
    if _score_cols_all:
        # 选 Top N 个最相关的评分列（按该簇均值绝对值排序）
        _score_means = {}
        for col in _score_cols_all:
            _score_means[col] = abs(float(adata.obs.loc[_mask, col].mean()))
        _top_scores_sorted = sorted(_score_means, key=_score_means.get, reverse=True)[:10]

        _fig, _axes = plt.subplots(
            max(1, len(_top_scores_sorted) // 5 + 1),
            min(5, len(_top_scores_sorted)),
            figsize=(min(5 * 3, 15), max(3, len(_top_scores_sorted) // 5 * 3)),
        )
        if isinstance(_axes, np.ndarray):
            _axes = _axes.flatten()
        else:
            _axes = [_axes]

        for _ax, _scol in zip(_axes, _top_scores_sorted):
            # 只画该簇细胞的评分分布
            _plot_df = pd.DataFrame({
                "cluster": [str(_cid)] * _n,
                "score": adata.obs.loc[_mask, _scol].values,
            })
            # 小提琴图：该簇细胞在该评分上的分布
            _parts = _ax.violinplot(
                _plot_df["score"].dropna(), positions=[0],
                showmeans=True, showmedians=True,
            )
            _ct_name = _scol.removeprefix("score_")
            _ax.set_title(_ct_name, fontsize=9)
            _ax.set_ylabel("Score")
            _ax.set_xticks([])
            _ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)

        # 隐藏多余子图
        for _ax in _axes[len(_top_scores_sorted):]:
            _ax.set_visible(False)

        _fig.suptitle(f"Gene set scores: {_cid} ({_n:,} cells)", fontsize=11)
        _fig.tight_layout()
        _fname = os.path.join(OUTPUT_DIR, f"scores_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)

        # 最高分 + 最低分的文字摘要
        _score_summary = []
        for col in _top_scores_sorted[:5]:
            _ct = col.removeprefix("score_")
            _mu = float(adata.obs.loc[_mask, col].mean())
            _pct = float((adata.obs.loc[_mask, col] > 0).mean() * 100)
            _score_summary.append(f"- {_ct}: mean={_mu:.3f}, {_pct:.1f}% cells positive")

        _md_lines.append(f"![Scores {_cid}](scores_{_cid_str}.png)\n\n")
        _md_lines.append("**评分摘要（Top 5）**:\n")
        _md_lines.append("\n".join(_score_summary) + "\n")
        print(f"  基因集评分: Top 5 = {_top_scores_sorted[:5]}")
    else:
        _md_lines.append("(无基因集评分列，跳过)\n")
        print("  基因集评分跳过: 无 score_* 列")

    # ---- 4. 跨疾病丰度柱状图 ----
    _md_lines.append(f"\n## 4. 跨疾病丰度\n")
    if _disease_available:
        # 该簇在各疾病组中的细胞比例
        _disease_counts = adata.obs.loc[_mask, DISEASE_COL].value_counts()
        _disease_total = adata.obs[DISEASE_COL].value_counts()
        _disease_pct = (_disease_counts / _disease_total * 100).dropna()

        _fig, _ax = plt.subplots(figsize=(max(6, len(_disease_pct)*0.8), 4))
        _bars = _ax.bar(range(len(_disease_pct)), _disease_pct.values, color="#3498db")
        _ax.set_xticks(range(len(_disease_pct)))
        _ax.set_xticklabels(_disease_pct.index, rotation=45, ha="right", fontsize=9)
        _ax.set_ylabel("% of disease group cells")
        _ax.set_title(f"Abundance of {_cid} across {DISEASE_COL}")
        # 在柱上标注百分比
        for _bar, _val in zip(_bars, _disease_pct.values):
            _ax.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 0.3,
                     f"{_val:.1f}%", ha="center", va="bottom", fontsize=8)
        _fig.tight_layout()
        _fname = os.path.join(OUTPUT_DIR, f"abundance_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)

        _md_lines.append(f"![Abundance {_cid}](abundance_{_cid_str}.png)\n\n")
        _md_lines.append(f"**{DISEASE_COL} 分布**:\n")
        for _d, _cnt in _disease_counts.items():
            _pct_val = _disease_pct.get(_d, 0)
            _md_lines.append(f"- {_d}: {_cnt} cells ({_pct_val:.1f}% of disease group)\n")
        print(f"  跨疾病丰度: {dict(_disease_counts)}")
    else:
        _md_lines.append(f"(obs 列 '{DISEASE_COL}' 不存在，跳过)\n")
        print(f"  跨疾病丰度跳过: DISEASE_COL='{DISEASE_COL}' 不在 obs")

    # ---- 暂时写入 markdown（不含 LLM 段落，LLM 段落单独追加）----
    with open(_out_md, "w") as _f:
        _f.writelines(_md_lines)
    print(f"  基础 markdown 已写入: {_out_md}")

    # 暂存摘要信息和 markdown 路径，供 LLM cell 使用
    _cluster_summaries.append({
        "cid": _cid,
        "cid_str": _cid_str,
        "n_cells": _n,
        "top_genes": _top_genes,
        "top_logfc": _top_logfc,
        "out_md": _out_md,
    })

print(f"\n{'='*60}")
print(f"逐簇基础剖析完成。共 {len(_cluster_summaries)} 个簇。")
print(f"{'='*60}")

## LLM 叙述段落（key 守卫——无 key 跳过）

对每个簇，将上述 UMUAP 位置、Top 标记基因、基因集评分、跨疾病丰度综合为
一份 prompt，调用 LLM 写一段 2-3 段的中文生物学叙述。

**为什么用 LLM 写叙述而非手写？** 每簇的证据维度多（marker 列表 +
评分 profile + 疾病分布），人工逐一综合费时且容易遗漏。
LLM 擅长把结构化证据转换为连贯叙述——但它是**初稿生成器**，
PI 仍需审阅修改后再用于论文。

**状态：代码已写，待 PI 在 `.env` 配 key 后人工运行调试。**
无 key 时优雅跳过，notebook 不崩溃。

调用模式沿用 `06_annotated.ipynb` 的 mLLMCelltype 模式：
按模型名前缀路由 provider（openai/anthropic/deepseek/qwen），
从 `.env` 取对应 `{PROVIDER}_API_KEY`。

In [ ]:
# LLM 叙述段落（key 守卫——无 key 跳过）。
# 为什么逐簇单独调用？每个簇的上下文（marker 列表 + 评分 + 疾病分布）
# 加起来已 ~500-1000 tokens，所有簇一起发给 LLM 容易超 context 且输出混杂。
# 逐簇调用虽然 API 调用次数多，但输出结构清晰，每簇叙述独立可审阅。

from dotenv import load_dotenv
load_dotenv()

_llm_providers = ["OPENAI", "ANTHROPIC", "DEEPSEEK", "QWEN", "GEMINI"]
_has_key = any(
    os.getenv(f"{p}_API_KEY") not in (None, "")
    for p in _llm_providers
)

if _has_key:
    _pfx = VERDICT_MODEL.split("/")[0].lower()
    if _pfx in ("openai", "qwen", "deepseek"):
        from openai import OpenAI
        _key_var = f"{_pfx.upper()}_API_KEY"
        _base_var = f"{_pfx.upper()}_BASE_URL"
        _key = os.getenv(_key_var)
        _base = os.getenv(_base_var) or None
        if not _key:
            print(f"{_pfx} API key 未配置，跳过 LLM 叙述")
        else:
            _client = OpenAI(api_key=_key, base_url=_base)

            # 为 prompt 准备全局上下文
            _disease_info = ""
            if _disease_available and DISEASE_COL in adata.obs.columns:
                _disease_counts = adata.obs[DISEASE_COL].value_counts().to_dict()
                _disease_info = f"各疾病组总细胞数: {json.dumps(_disease_counts, ensure_ascii=False)}"

            for _cs in _cluster_summaries:
                _cid = _cs["cid"]
                _cid_str = _cs["cid_str"]
                _genes_list = ", ".join(_cs["top_genes"])
                _prompt = (
                    f"你是单细胞转录组学与胃粘膜生物学专家。请为以下细胞簇写一段中文叙述。\n\n"
                    f"## 上下文\n"
                    f"- 物种: human\n"
                    f"- 组织: stomach（胃粘膜）\n"
                    f"- 簇标签: {_cid}\n"
                    f"- 细胞数: {_cs['n_cells']:,}\n\n"
                    f"## Top 差异标记基因（该簇 vs 其余簇，Wilcoxon）\n"
                    f"{_genes_list}\n\n"
                    f"{_disease_info}\n\n"
                    f"## 任务\n"
                    f"1. 根据标记基因判断该簇最可能的细胞类型（具体到亚型）\n"
                    f"2. 解释为什么这些标记基因支持这个判断（2-3 个关键基因的功能）\n"
                    f"3. 如果存在跨疾病丰度差异，分析可能的生物学意义\n"
                    f"4. 提出 1-2 个该簇值得进一步研究的生物学问题\n\n"
                    f"用中文写 2-3 段，结构清晰，每段有明确的主题句。"
                )

                try:
                    _resp = _client.chat.completions.create(
                        model=VERDICT_MODEL,
                        messages=[
                            {"role": "system",
                             "content": "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱注释。"},
                            {"role": "user", "content": _prompt},
                        ],
                        temperature=0.3,
                        max_tokens=1200,
                    )
                    _narrative = _resp.choices[0].message.content
                except Exception as _e:
                    _narrative = f"(LLM 调用失败: {_e})"

                # 追加到已存的 markdown 文件末尾
                with open(_cs["out_md"], "a") as _f:
                    _f.write(f"\n## 5. LLM 综合叙述\n\n")
                    _f.write(f"_模型: {VERDICT_MODEL}_\n\n")
                    _f.write(_narrative + "\n")
                print(f"  簇 {_cid}: LLM 叙述已追加 -> {_cs['out_md']}")
    elif _pfx == "anthropic":
        from anthropic import Anthropic
        _key = os.getenv("ANTHROPIC_API_KEY")
        _base = os.getenv("ANTHROPIC_BASE_URL") or None
        if not _key:
            print("Anthropic API key 未配置，跳过 LLM 叙述")
        else:
            print(
                "Anthropic 叙述模式（代码结构同 OpenAI 模式）——"
                "待 PI 实现或改用 OpenAI 兼容 endpoint"
            )
    else:
        print(
            f"VERDICT_MODEL 的 provider '{_pfx}' 暂未支持。"
            "请在 PARAMS 选 openai/anthropic/deepseek/qwen"
        )
else:
    print("=" * 60)
    print("LLM 叙述段落已跳过——未检测到 LLM API key。")
    print("请在项目根目录 .env 文件中配置至少一家 provider 的 API key，")
    print("然后重新运行本 cell。模板见 .env.example。")
    print("=" * 60)
    # 为每个 cluster markdown 追加"LLM 未运行"说明
    for _cs in _cluster_summaries:
        with open(_cs["out_md"], "a") as _f:
            _f.write(
                f"\n## 5. LLM 综合叙述\n\n"
                f"_(LLM 未运行——请配置 .env 中的 API key 后重新执行上方 cell)_\n"
            )

## 生成 index.md

将所有逐簇 markdown 串成索引页，方便 PI 快速浏览和跳转。

In [ ]:
# 生成 index.md——将所有逐簇分析页串成一个总表。
_index_path = os.path.join(OUTPUT_DIR, "index.md")
_index_lines = []
_index_lines.append("# Per-Cluster Deep Profile 索引\n\n")
_index_lines.append(f"**标签列**: `{LABEL_COL}`\n")
_index_lines.append(f"**上游**: `{UPSTREAM_PATH}`\n")
_index_lines.append(f"**总细胞数**: {adata.n_obs:,}\n")
_index_lines.append(f"**输出簇数**: {len(_cluster_summaries)}\n\n")
_index_lines.append("| 簇标签 | 细胞数 | 占比 | Top 3 标记基因 | 链接 |\n")
_index_lines.append("|--------|--------|------|----------------|------|\n")

for _cs in _cluster_summaries:
    _pct = _cs["n_cells"] / adata.n_obs * 100
    # Top 3 基因（取前三个或更少）
    _top3 = _cs["top_genes"][:3] if len(_cs["top_genes"]) >= 3 else _cs["top_genes"]
    _top3_str = ", ".join(str(g) for g in _top3)
    _link = f"cluster_{_cs['cid_str']}.md"
    _index_lines.append(
        f"| {_cs['cid']} | {_cs['n_cells']:,} | {_pct:.1f}% | {_top3_str} | [{_cs['cid']}]({_link}) |\n"
    )

with open(_index_path, "w") as _f:
    _f.writelines(_index_lines)
print(f"索引页已写出: {_index_path}")
print(f"包含 {len(_cluster_summaries)} 个簇")

## 注：逐簇分析只产生 markdown 产物，不写新 h5ad

`06b_per_cluster` 是只读分析——它消费上游 `06_annotated` 的 h5ad，
产出逐簇 markdown 报告，不修改底层数据。因此不需要 `adata.write_h5ad`。
下面仅做内存释放。

In [ ]:
# 内存纪律自检——只读分析不改变 X 结构，做一个轻量断言即可。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 在只读分析过程中被意外改变: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 保持 sparse CSR float32")

# 释放内存（只读分析不写 h5ad，直接释放即可）
# 为什么不写 h5ad？本 notebook 不产生新数据列——只消费已有 obs/obsm
# 产出是 markdown 文件，已全部写到 OUTPUT_DIR
del adata
del _cluster_summaries
del _markers
gc.collect()
print("内存已释放")